# L4: Clasificarea florilor cu PyTorch

Setul de date: **tf_flowers** (3670 de poze cu 5 tipuri de flori: daisy, dandelion, roses, sunflowers, tulips).

**Cum se ruleaza:**
- Intai `MOD_RAPID = True` in prima celula si **Run All**: un test de cateva minute, ca sa vedem ca merge totul.
- Apoi `MOD_RAPID = False` si **Run All**: toate cele 14 experimente, cu 5-fold cross-validation. Poate dura una-doua ore pe GPU; se poate lasa peste noapte.
- Fiecare experiment terminat se salveaza, deci rularea poate fi oprita si reluata.

In [ ]:
# ======================================================================
# LAB4 - PAS 1: IMPORTURI SI SETARI
# - MOD_RAPID = True ruleaza un test scurt (putine imagini, 2 fold-uri, 2 epoci), ca sa vedeti ca merge totul
# - MOD_RAPID = False ruleaza experimentele complete (5 fold-uri, pana la 40 de epoci)
# =========================================================================
import os
import json
import copy
import time
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets.utils import download_and_extract_archive
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

MOD_RAPID = False

SEED = 42                      # acelasi seed peste tot, ca experimentele sa fie comparabile
MARIME_IMAGINE = 64            # imaginile sunt micsorate la 64x64 pixeli (downscaling, cum permite enuntul)
BATCH = 64                     # cate imagini intra odata in retea
RABDARE = 5                    # early stopping: ne oprim daca loss-ul de validare nu scade 5 epoci la rand
NR_FOLDURI = 2 if MOD_RAPID else 5
EPOCI_MAX = 2 if MOD_RAPID else 40

DIR_DATE = 'data'                                            # aici se descarca setul de date
DIR_REZULTATE = 'rezultate_L4' + ('_rapid' if MOD_RAPID else '')   # grafice, istoric, tabel cu rezultate
DIR_MODELE = 'modele_L4' + ('_rapid' if MOD_RAPID else '')         # modelele salvate
os.makedirs(DIR_REZULTATE, exist_ok=True)
os.makedirs(DIR_MODELE, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("PyTorch:", torch.__version__)
print("Antrenam pe:", device)
print("Procesor:", platform.processor())
if torch.cuda.is_available():
    print("Placa video:", torch.cuda.get_device_name(0))


def log(mesaj):
    # afiseaza mesajul si il scrie si in fisierul log.txt (cu ora), ca sa ramana o urma a rularii
    print(mesaj)
    with open(os.path.join(DIR_REZULTATE, 'log.txt'), 'a', encoding='utf-8') as f:
        f.write(time.strftime('%Y-%m-%d %H:%M:%S') + '  ' + mesaj + '\n')

In [ ]:
# ======================================================================
# LAB4 - PAS 2: SETUL DE DATE (tf_flowers / flower_photos)
# - la prima rulare se descarca arhiva (aprox. 220 MB) in folderul data/
# - imaginile se micsoreaza la 64x64 si se salveaza intr-un fisier cache, ca rularile urmatoare sa fie rapide
# =========================================================================
URL = 'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz'
dir_flori = os.path.join(DIR_DATE, 'flower_photos')
if not os.path.isdir(dir_flori):
    download_and_extract_archive(URL, download_root=DIR_DATE)

clase = sorted(d for d in os.listdir(dir_flori) if os.path.isdir(os.path.join(dir_flori, d)))
fisier_cache = os.path.join(DIR_DATE, f'flori_{MARIME_IMAGINE}px.pt')

if os.path.exists(fisier_cache):
    imagini, etichete, cai = torch.load(fisier_cache)
else:
    lista_imagini, lista_etichete, cai = [], [], []
    for i, clasa in enumerate(clase):
        for nume_fisier in sorted(os.listdir(os.path.join(dir_flori, clasa))):
            if not nume_fisier.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            cale = os.path.join(dir_flori, clasa, nume_fisier)
            img = Image.open(cale).convert('RGB').resize((MARIME_IMAGINE, MARIME_IMAGINE))
            # din (inaltime, latime, 3 culori) in (3 culori, inaltime, latime), cum vrea PyTorch
            lista_imagini.append(torch.from_numpy(np.array(img)).permute(2, 0, 1))
            lista_etichete.append(i)
            cai.append(cale)
    imagini = torch.stack(lista_imagini)        # valori intregi 0-255, forma (N, 3, 64, 64)
    etichete = torch.tensor(lista_etichete)
    torch.save((imagini, etichete, cai), fisier_cache)

if MOD_RAPID:
    # test rapid: doar primele 40 de imagini din fiecare clasa
    idx = torch.cat([torch.nonzero(etichete == c).flatten()[:40] for c in range(len(clase))])
    imagini, etichete, cai = imagini[idx], etichete[idx], [cai[i] for i in idx.tolist()]

print("Clase:", clase)
print("Forma tensorului cu imagini:", tuple(imagini.shape))
print(pd.Series(etichete.numpy()).map(dict(enumerate(clase))).value_counts().to_string())

# cate o imagine din fiecare clasa
fig, axe = plt.subplots(1, len(clase), figsize=(3 * len(clase), 3))
for c, ax in enumerate(axe):
    i = torch.nonzero(etichete == c).flatten()[0].item()
    ax.imshow(imagini[i].permute(1, 2, 0).numpy())
    ax.set_title(clase[c])
    ax.axis('off')
plt.show()

### Interpretare: setul de date (de completat dupa rulare)

- Cate poze are fiecare clasa? Sunt clasele echilibrate sau exista o clasa mult mai mica (ca la clasele de pret din L2)?
- De ce folosim macro-F1, si nu accuracy, daca clasele nu au acelasi numar de poze?
- Ce se pierde cand micsoram pozele la 64x64? Ce detalii ale florilor raman vizibile?

In [ ]:
# ======================================================================
# LAB4 - PAS 3: CUM AJUNG IMAGINILE IN RETEA (Dataset, normalizare, augmentare)
# - normalizare: valorile pixelilor ajung intre -1 si 1
# - augmentare (doar la antrenare, daca experimentul o cere): oglindire stanga-dreapta si rotire usoara
# =========================================================================
class SetFlori(Dataset):
    def __init__(self, imagini, etichete, augmentare=False):
        self.imagini = imagini
        self.etichete = etichete
        if augmentare:
            self.augmentare = transforms.Compose([transforms.RandomHorizontalFlip(),
                                                  transforms.RandomRotation(15)])
        else:
            self.augmentare = None

    def __len__(self):
        return len(self.etichete)

    def __getitem__(self, i):
        x = self.imagini[i].float() / 255.0          # 0-255 -> 0-1
        if self.augmentare is not None:
            x = self.augmentare(x)
        x = (x - 0.5) / 0.5                          # 0-1 -> -1..1
        return x, self.etichete[i]

In [ ]:
# ======================================================================
# LAB4 - PAS 4: RETEAUA NEURONALA, CONSTRUITA DIN SETARILE EXPERIMENTULUI
# - straturi convolutionale (0-6), fiecare cu: convolutie 3x3 -> [batch norm] -> activare -> [max pooling]
# - [optional] un strat recurent (GRU) care citeste imaginea rand cu rand
# - straturi ascunse complet conectate (1-3), fiecare cu: liniar -> [batch norm] -> activare -> dropout
# - stratul de iesire: cate un neuron pentru fiecare clasa de flori
# =========================================================================
ACTIVARI = {'relu': nn.ReLU, 'leaky_relu': nn.LeakyReLU, 'tanh': nn.Tanh}


class ReteaFlori(nn.Module):
    def __init__(self, cfg, nr_clase):
        super().__init__()
        activare = ACTIVARI[cfg['activare']]

        # partea convolutionala: "cauta" forme si culori in imagine
        straturi = []
        canale, marime = 3, MARIME_IMAGINE
        for filtre in cfg['conv']:
            straturi.append(nn.Conv2d(canale, filtre, kernel_size=3, padding=1))
            if cfg['batch_norm']:
                straturi.append(nn.BatchNorm2d(filtre))
            straturi.append(activare())
            if cfg['pooling']:
                straturi.append(nn.MaxPool2d(2))     # injumatateste inaltimea si latimea
                marime //= 2
            canale = filtre
        self.conv = nn.Sequential(*straturi)

        # partea recurenta (optional): fiecare rand al imaginii devine un pas dintr-o secventa
        self.recurent = cfg['recurent']
        if self.recurent:
            self.gru = nn.GRU(input_size=canale * marime, hidden_size=128, batch_first=True)
            intrari = 128
        else:
            intrari = canale * marime * marime

        # partea complet conectata: ia decizia finala
        dense = []
        for neuroni in cfg['hidden']:
            dense.append(nn.Linear(intrari, neuroni))
            if cfg['batch_norm']:
                dense.append(nn.BatchNorm1d(neuroni))
            dense.append(activare())
            dense.append(nn.Dropout(cfg['dropout']))
            intrari = neuroni
        dense.append(nn.Linear(intrari, nr_clase))
        self.dense = nn.Sequential(*dense)

    def forward(self, x):
        x = self.conv(x)
        if self.recurent:
            # (batch, canale, inaltime, latime) -> (batch, inaltime, canale * latime)
            x = x.permute(0, 2, 1, 3).flatten(2)
            _, ultima_stare = self.gru(x)
            x = ultima_stare[-1]
        else:
            x = x.flatten(1)                         # "intindem" totul intr-un singur vector
        return self.dense(x)


def numara_parametri(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
# ======================================================================
# LAB4 - PAS 5: EXPERIMENTELE (in stilul grid search)
# - pornim de la o retea de baza si schimbam cate un lucru odata, ca sa vedem efectul fiecarei setari
# - acopera tot ce cere enuntul: 1-3 straturi ascunse, 0-6 straturi convolutionale, cu/fara pooling,
#   mai multe numere de neuroni, activari, rate de invatare, L1, L2, batch norm, dropout, augmentare, recurenta
# =========================================================================
BAZA = dict(conv=[16, 32, 64, 128], pooling=True, hidden=[128], activare='relu', lr=1e-3,
            l1=0.0, l2=0.0, batch_norm=False, dropout=0.0, augmentare=False, recurent=False)


def experiment(nume, **schimbari):
    cfg = dict(BAZA)
    cfg.update(schimbari)
    cfg['nume'] = nume
    return cfg


EXPERIMENTE = [
    experiment('01_fara_conv', conv=[], hidden=[256]),
    experiment('02_conv2_pool', conv=[16, 32]),
    experiment('03_conv2_fara_pool', conv=[16, 32], pooling=False),
    experiment('04_conv4_baza'),
    experiment('05_conv6', conv=[16, 32, 64, 128, 128, 128]),
    experiment('06_3_ascunse', hidden=[256, 128, 64]),
    experiment('07_tanh', activare='tanh'),
    experiment('08_leaky_lr_mic', activare='leaky_relu', lr=1e-4),
    experiment('09_lr_mare', lr=1e-2),
    experiment('10_l2', l2=1e-3),
    experiment('11_l1', l1=1e-5),
    experiment('12_bn_dropout05', batch_norm=True, dropout=0.5),
    experiment('13_dropout025_augm', dropout=0.25, augmentare=True),
    experiment('14_conv_gru', conv=[16, 32], recurent=True),
]

tabel = pd.DataFrame(EXPERIMENTE).set_index('nume')
tabel['parametri'] = [numara_parametri(ReteaFlori(cfg, len(clase))) for cfg in EXPERIMENTE]
print(tabel.to_string())

### Interpretare: experimentele (de completat dupa rulare)

- Care experiment are cei mai multi parametri si de ce? (Indiciu: uitati-va la cel fara pooling.)
- Cum schimba pooling-ul numarul de parametri?
- Are cel cu 6 straturi convolutionale mai multi sau mai putini parametri decat cel cu 4? De ce?

In [ ]:
# ======================================================================
# LAB4 - PAS 6: ANTRENAREA UNUI MODEL (cu early stopping, L1, L2)
# - Adam cu weight_decay = regularizare L2
# - L1 se adauga manual la loss: suma valorilor absolute ale tuturor ponderilor
# - dupa fiecare epoca: loss si macro-F1 pe antrenare si pe validare
# - early stopping: pastram modelul din epoca cu cel mai mic loss de validare
# =========================================================================
def evalueaza(model, dl, loss_fn, device):
    model.eval()
    total, predictii, adevarate = 0.0, [], []
    with torch.no_grad():
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            iesire = model(x)
            total += loss_fn(iesire, y).item() * len(y)
            predictii += iesire.argmax(1).tolist()
            adevarate += y.tolist()
    return total / len(adevarate), predictii, adevarate


def antreneaza(cfg, idx_antr, idx_val, device, epoci_max=EPOCI_MAX):
    torch.manual_seed(SEED)
    model = ReteaFlori(cfg, len(clase)).to(device)
    optimizator = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['l2'])
    loss_fn = nn.CrossEntropyLoss()
    # drop_last: ultimul batch, daca e incomplet, e sarit (batch norm nu merge cu un batch de o singura poza)
    dl_antr = DataLoader(SetFlori(imagini[idx_antr], etichete[idx_antr], cfg['augmentare']),
                         batch_size=BATCH, shuffle=True, drop_last=True)
    dl_val = DataLoader(SetFlori(imagini[idx_val], etichete[idx_val]), batch_size=BATCH)

    istoric = {'loss_antr': [], 'loss_val': [], 'f1_antr': [], 'f1_val': []}
    cel_mai_bun = {'loss': float('inf'), 'stare': None, 'pred': None, 'epoca': 0}
    epoci_fara_progres = 0
    start = time.time()

    for epoca in range(epoci_max):
        model.train()
        total, predictii, adevarate = 0.0, [], []
        for x, y in dl_antr:
            x, y = x.to(device), y.to(device)
            iesire = model(x)
            loss = loss_fn(iesire, y)
            if cfg['l1'] > 0:
                loss = loss + cfg['l1'] * sum(p.abs().sum() for p in model.parameters())
            optimizator.zero_grad()
            loss.backward()
            optimizator.step()
            total += loss.item() * len(y)
            predictii += iesire.argmax(1).tolist()
            adevarate += y.tolist()

        loss_val, pred_val, adev_val = evalueaza(model, dl_val, loss_fn, device)
        istoric['loss_antr'].append(total / len(adevarate))
        istoric['loss_val'].append(loss_val)
        istoric['f1_antr'].append(f1_score(adevarate, predictii, average='macro'))
        istoric['f1_val'].append(f1_score(adev_val, pred_val, average='macro'))

        if loss_val < cel_mai_bun['loss']:
            cel_mai_bun = {'loss': loss_val, 'stare': copy.deepcopy(model.state_dict()),
                           'pred': pred_val, 'epoca': epoca + 1}
            epoci_fara_progres = 0
        else:
            epoci_fara_progres += 1
            if epoci_fara_progres >= RABDARE:
                break                                # early stopping

    durata = time.time() - start
    model.load_state_dict(cel_mai_bun['stare'])
    return model, istoric, cel_mai_bun, durata

In [ ]:
# ======================================================================
# LAB4 - PAS 7: RULAREA TUTUROR EXPERIMENTELOR CU 5-FOLD CROSS-VALIDATION
# - aceleasi 5 fold-uri (acelasi random seed) pentru toate experimentele
# - pentru fiecare experiment se salveaza: predictiile, istoricul pe epoci, modelul din primul fold
# - rezultatele se adauga in rezultate.csv; un experiment deja rulat este sarit,
#   deci celula poate fi oprita si pornita din nou (sau lasata peste noapte)
# =========================================================================
skf = StratifiedKFold(n_splits=NR_FOLDURI, shuffle=True, random_state=SEED)
folduri = list(skf.split(np.zeros(len(etichete)), etichete.numpy()))
fisier_rezultate = os.path.join(DIR_REZULTATE, 'rezultate.csv')

for cfg in EXPERIMENTE:
    nume = cfg['nume']
    dir_exp = os.path.join(DIR_REZULTATE, nume)
    if os.path.exists(os.path.join(dir_exp, 'pred_oof.npy')):
        log(f"{nume}: deja rulat, il sarim")
        continue
    os.makedirs(dir_exp, exist_ok=True)
    log(f"=== {nume} ===")

    # predictii "out-of-fold": fiecare imagine e prezisa de modelul care NU a vazut-o la antrenare
    pred_oof = np.full(len(etichete), -1)
    istorice, f1_folduri, durate, epoci = [], [], [], []
    for k, (idx_antr, idx_val) in enumerate(folduri):
        model, istoric, cel_mai_bun, durata = antreneaza(cfg, idx_antr, idx_val, device)
        pred_oof[idx_val] = cel_mai_bun['pred']
        f1_folduri.append(f1_score(etichete[idx_val].numpy(), cel_mai_bun['pred'], average='macro'))
        istorice.append(istoric)
        durate.append(durata)
        epoci.append(cel_mai_bun['epoca'])
        if k == 0:
            cale_model = os.path.join(DIR_MODELE, f'{nume}.pt')
            torch.save(model.state_dict(), cale_model)
        log(f"  fold {k + 1}: macro-F1 {f1_folduri[-1]:.3f} | cea mai buna epoca {cel_mai_bun['epoca']} "
            f"| {len(istoric['loss_val'])} epoci rulate | {durata:.0f} s")

    np.save(os.path.join(dir_exp, 'pred_oof.npy'), pred_oof)
    with open(os.path.join(dir_exp, 'istoric.json'), 'w') as f:
        json.dump(istorice, f)

    rand = {'experiment': nume,
            'macro_f1_medie': round(float(np.mean(f1_folduri)), 4),
            'macro_f1_std': round(float(np.std(f1_folduri)), 4),
            'parametri': numara_parametri(model),
            'timp_antrenare_s': round(sum(durate), 1),
            'epoca_optima_medie': round(float(np.mean(epoci)), 1),
            'marime_model_MB': round(os.path.getsize(cale_model) / 1e6, 2),
            'device': str(device)}
    rezultate = pd.read_csv(fisier_rezultate) if os.path.exists(fisier_rezultate) else pd.DataFrame()
    rezultate = pd.concat([rezultate, pd.DataFrame([rand])], ignore_index=True)
    rezultate.to_csv(fisier_rezultate, index=False)
    log(f"  {nume}: macro-F1 {rand['macro_f1_medie']:.3f} +/- {rand['macro_f1_std']:.3f}")

In [ ]:
# ======================================================================
# LAB4 - PAS 8: GRAFICE PENTRU FIECARE EXPERIMENT
# - loss pe antrenare / validare pe epoci (cate o linie pentru fiecare fold)
# - macro-F1 pe antrenare / validare pe epoci
# - matricea de confuzie, din predictiile out-of-fold (toate imaginile)
# =========================================================================
for cfg in EXPERIMENTE:
    nume = cfg['nume']
    dir_exp = os.path.join(DIR_REZULTATE, nume)
    if not os.path.exists(os.path.join(dir_exp, 'pred_oof.npy')):
        continue
    with open(os.path.join(dir_exp, 'istoric.json')) as f:
        istorice = json.load(f)
    pred_oof = np.load(os.path.join(dir_exp, 'pred_oof.npy'))

    fig, axe = plt.subplots(1, 3, figsize=(18, 4.5))
    for k, ist in enumerate(istorice):
        ep = range(1, len(ist['loss_antr']) + 1)
        axe[0].plot(ep, ist['loss_antr'], color='tab:blue', alpha=0.6, label='antrenare' if k == 0 else None)
        axe[0].plot(ep, ist['loss_val'], color='tab:orange', alpha=0.6, label='validare' if k == 0 else None)
        axe[1].plot(ep, ist['f1_antr'], color='tab:blue', alpha=0.6, label='antrenare' if k == 0 else None)
        axe[1].plot(ep, ist['f1_val'], color='tab:orange', alpha=0.6, label='validare' if k == 0 else None)
    axe[0].set_title('Loss pe epoci')
    axe[0].set_xlabel('Epoca')
    axe[0].legend()
    axe[1].set_title('Macro-F1 pe epoci')
    axe[1].set_xlabel('Epoca')
    axe[1].legend()
    ConfusionMatrixDisplay(confusion_matrix(etichete.numpy(), pred_oof),
                           display_labels=clase).plot(ax=axe[2], cmap='Blues', colorbar=False)
    axe[2].set_title('Matricea de confuzie')
    plt.setp(axe[2].get_xticklabels(), rotation=30)
    fig.suptitle(f"{nume}  (macro-F1 out-of-fold: {f1_score(etichete.numpy(), pred_oof, average='macro'):.3f})")
    plt.tight_layout()
    plt.savefig(os.path.join(dir_exp, 'grafice.png'), dpi=80)
    plt.show()

### Interpretare: graficele pe epoci si matricile de confuzie (de completat dupa rulare)

Ce urmariti in fiecare grafic:
- **Overfitting:** loss-ul de antrenare continua sa scada, dar cel de validare se opreste sau creste; macro-F1 pe antrenare urca spre 1, iar pe validare ramane mult mai jos. La ce experimente se vede cel mai tare?
- **Underfitting:** ambele curbe raman sus (loss) sau jos (macro-F1). Apare la vreun experiment (de exemplu fara convolutii sau cu rata de invatare mica)?
- **Rata de invatare mare:** curbe care sar in sus si in jos in loc sa scada lin.
- **Regularizare, dropout, augmentare:** reduc diferenta dintre antrenare si validare?
- **Matricea de confuzie:** ce flori se confunda cel mai des intre ele? (De exemplu trandafiri cu lalele, care au culori asemanatoare.) Ce floare e recunoscuta cel mai bine?

In [ ]:
# ======================================================================
# LAB4 - PAS 9: COMPARATIA EXPERIMENTELOR
# - calitatea (macro-F1), numarul de parametri, timpul de antrenare, marimea modelului pe disc
# =========================================================================
rezultate = pd.read_csv(fisier_rezultate).sort_values('macro_f1_medie', ascending=False)
print(rezultate.to_string(index=False))

fig, axe = plt.subplots(1, 3, figsize=(20, 5))
axe[0].barh(rezultate['experiment'], rezultate['macro_f1_medie'], xerr=rezultate['macro_f1_std'], color='seagreen')
axe[0].set_title('Macro-F1 mediu pe 5 fold-uri')
axe[0].invert_yaxis()
axe[1].scatter(rezultate['parametri'], rezultate['macro_f1_medie'])
for _, r in rezultate.iterrows():
    axe[1].annotate(r['experiment'][:2], (r['parametri'], r['macro_f1_medie']))
axe[1].set_xscale('log')
axe[1].set_title('Macro-F1 in functie de numarul de parametri')
axe[1].set_xlabel('Parametri (scara logaritmica)')
axe[2].barh(rezultate['experiment'], rezultate['timp_antrenare_s'], color='steelblue')
axe[2].set_title('Timpul total de antrenare (s)')
axe[2].invert_yaxis()
plt.tight_layout()
plt.show()

print("\nMarimea modelelor salvate pe disc:")
for nume_fisier in sorted(os.listdir(DIR_MODELE)):
    print(f"  {nume_fisier:30s} {os.path.getsize(os.path.join(DIR_MODELE, nume_fisier)) / 1e6:8.2f} MB")

### Interpretare: comparatia experimentelor (de completat dupa rulare)

- Care experiment are cel mai bun macro-F1? Diferenta fata de urmatoarele e mai mare decat abaterea standard sau poate fi intamplatoare (ca la L2)?
- **Cum influenteaza hiperparametrii calitatea?** Ajuta convolutiile (01 vs restul)? Ajuta mai multe straturi convolutionale (02, 04, 05)? Pooling-ul (02 vs 03)? Activarile (07, 08)? Rata de invatare (08, 09)? Regularizarea (10, 11, 12, 13)? Recurenta (14)?
- **Cum influenteaza hiperparametrii timpul de antrenare?** Care experimente dureaza cel mai mult si de ce (parametri multi, fara pooling, augmentare)?
- **Marimea pe disc:** este proportionala cu numarul de parametri? Care model e cel mai mare si de ce?

In [ ]:
# ======================================================================
# LAB4 - PAS 10: DIMENSIUNILE VECTORILOR SI MATRICELOR PENTRU CEL MAI BUN MODEL
# - forma iesirii fiecarui strat, pentru o singura imagine
# - forma fiecarei matrice de ponderi si numarul total de parametri
# =========================================================================
nume_best = rezultate.iloc[0]['experiment']
cfg_best = next(c for c in EXPERIMENTE if c['nume'] == nume_best)
model_best = ReteaFlori(cfg_best, len(clase))
model_best.load_state_dict(torch.load(os.path.join(DIR_MODELE, f'{nume_best}.pt'), map_location='cpu'))
model_best.eval()
print("Cel mai bun model:", nume_best)


def arata_forma(strat, intrare, iesire):
    if isinstance(iesire, tuple):                    # GRU intoarce (iesiri, ultima stare)
        iesire = iesire[0]
    print(f"  {strat.__class__.__name__:12s} -> {tuple(iesire.shape)}")


carlige = [m.register_forward_hook(arata_forma) for m in model_best.modules() if len(list(m.children())) == 0]
x = torch.zeros(1, 3, MARIME_IMAGINE, MARIME_IMAGINE)
print("Intrare:", tuple(x.shape))
with torch.no_grad():
    model_best(x)
for c in carlige:
    c.remove()

print("\nPonderile (matricele) fiecarui strat:")
for nume_p, p in model_best.named_parameters():
    print(f"  {nume_p:25s} {str(tuple(p.shape)):20s} {p.numel():>10,} parametri")
print("Total parametri:", f"{numara_parametri(model_best):,}")

### Interpretare: dimensiunile celui mai bun model (de completat dupa rulare)

- Cum se schimba forma imaginii de la un strat la altul? Ce face pooling-ul cu inaltimea si latimea? Ce face convolutia cu numarul de canale?
- Ce strat are cei mai multi parametri? (De obicei primul strat complet conectat, cel de dupa convolutii.)

In [ ]:
# ======================================================================
# LAB4 - PAS 11: POZELE CLASIFICATE GRESIT DE CELE MAI MULTE MODELE
# - pentru fiecare imagine numaram cate experimente au gresit-o (din predictiile out-of-fold)
# =========================================================================
toate_pred = {}
for cfg in EXPERIMENTE:
    cale = os.path.join(DIR_REZULTATE, cfg['nume'], 'pred_oof.npy')
    if os.path.exists(cale):
        toate_pred[cfg['nume']] = np.load(cale)
matrice_pred = np.array(list(toate_pred.values()))           # (experimente, imagini)
nr_gresite = (matrice_pred != etichete.numpy()).sum(axis=0)
cele_mai_gresite = np.argsort(-nr_gresite)[:8]

fig, axe = plt.subplots(2, 4, figsize=(16, 8))
for ax, i in zip(axe.ravel(), cele_mai_gresite):
    pred_frecventa = np.bincount(matrice_pred[:, i], minlength=len(clase)).argmax()
    ax.imshow(Image.open(cai[i]).convert('RGB'))
    ax.set_title(f"adevarat: {clase[int(etichete[i])]}\nprezis des: {clase[pred_frecventa]}\n"
                 f"gresit de {nr_gresite[i]} din {len(toate_pred)} modele", fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Interpretare: pozele clasificate gresit (de completat dupa rulare)

Pentru fiecare poza, incercati sa explicati de ce e grea. Motive frecvente:
- floarea e mica in poza sau e acoperita de alte lucruri (oameni, frunze, cer);
- in poza sunt mai multe tipuri de flori;
- culori neobisnuite, lumina slaba sau poza foarte apropiata;
- doua tipuri de flori care seamana (de exemplu trandafiri si lalele de aceeasi culoare);
- la 64x64 pixeli se pierd detaliile care ar face diferenta;
- uneori chiar eticheta din setul de date pare gresita.

In [ ]:
# ======================================================================
# LAB4 - PAS 12: TIMPUL DE ANTRENARE CPU vs GPU
# - 3 experimente, un singur fold, 5 epoci fixe, ca sa comparam doar viteza
# =========================================================================
if torch.cuda.is_available():
    comparatie = []
    idx_antr, idx_val = folduri[0]
    for nume in ['02_conv2_pool', '04_conv4_baza', '12_bn_dropout05']:
        cfg = next(c for c in EXPERIMENTE if c['nume'] == nume)
        rand = {'experiment': nume}
        for dev in [torch.device('cpu'), torch.device('cuda')]:
            _, _, _, durata = antreneaza(cfg, idx_antr, idx_val, dev, epoci_max=5)
            rand[f'timp_{dev.type}_s'] = round(durata, 1)
        rand['de_cate_ori_mai_rapid_pe_GPU'] = round(rand['timp_cpu_s'] / rand['timp_cuda_s'], 1)
        comparatie.append(rand)
        log(f"CPU vs GPU {nume}: {rand}")
    comparatie = pd.DataFrame(comparatie)
    comparatie.to_csv(os.path.join(DIR_REZULTATE, 'cpu_vs_gpu.csv'), index=False)
    print("Procesor:", platform.processor())
    print("Placa video:", torch.cuda.get_device_name(0))
    print(comparatie.to_string(index=False))
    comparatie.set_index('experiment')[['timp_cpu_s', 'timp_cuda_s']].plot(kind='bar', figsize=(9, 4), rot=0)
    plt.title('Timp de antrenare pentru 5 epoci: CPU vs GPU')
    plt.ylabel('secunde')
    plt.show()
else:
    print("Nu exista o placa video NVIDIA disponibila, deci comparatia CPU vs GPU nu se poate face pe acest calculator.")

### Interpretare: CPU vs GPU (de completat dupa rulare)

- De cate ori e mai rapid GPU-ul? Diferenta e aceeasi la toate experimentele sau creste la retelele mai mari?
- De ce e GPU-ul mai rapid? (Cursul 5: antrenarea inseamna multe inmultiri de matrice, pe care GPU-ul le face in paralel.)
- Procesor si placa video folosite: (completati din rezultatul celulei).

## Concluzie L4 (de completat dupa rulare)

- Cel mai bun model si macro-F1-ul lui:
- Ce a ajutat cel mai mult (convolutii, pooling, regularizare, augmentare...):
- Ce nu a ajutat sau a stricat:
- Ce flori se confunda cel mai des si de ce:
- Cat de mult a accelerat GPU-ul antrenarea: